In [ ]:
from pylab import *

# Resolution

In [ ]:
N = np.logspace(7, 10, 12, base = 2.0, dtype = int)
print(N)

In [ ]:
rr_inf = 1.0 * (N[0] + 3 - 2 + 0.5)
rr_inf

In [ ]:
dr = rr_inf / (N + 1 + 0.5)
print(dr)

In [ ]:
["%.5E and %04d" % (dr[i], N[i]) for i in range(len(N))]

In [ ]:
for i in range(1, len(N)):
    par_name = "/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d.par" % (dr[i], N[i])
    
    f = open(par_name, "w")
    
    par_file = """
# GRID
dr 		= %.16E
dz 		= %.16E
NrInterior	= %d
NzInterior	= %d
order 		= 4

# SCALAR FIELD PROPERTIES
l	 = 2
m 	 = 1.0

# INITIAL FREQUENCY.
#w0 	= 7.50000E-01

# INITIAL DATA
readInitialData	= 3
NrTotalInitial 	= %d
NzTotalInitial 	= %d
dr_i		= %.16E
dz_i		= %.16E
ghost_i		= 2
order_i		= 4
log_alpha_i 	= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/log_alpha_f.asc"
beta_i		= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/beta_f.asc"
log_h_i		= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/log_h_f.asc"
log_a_i		= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/log_a_f.asc"
psi_i		= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/psi_f.asc"
lambda_i	= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/lambda_f.asc"
w_i		= "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=7.50000E-01,dr=%.5E,N=%04d/w_f.asc"

# ANALYTIC INITIAL DATA PARAMETERS.
psi0 	= 1.000
sigmaR	= 0.0
sigmaZ	= 0.0
rExt	= 0.0

# FIXED VARIABLE.
fixedPhi  	= 0
fixedPhiR 	= 0
fixedPhiZ	= 0
fixedOmega 	= 1

# SOLVER PARAMETERS.
solverType	= 1
localSolver	= 1
epsilon		= 2.0E-13
maxNewtonIter   = 50
lambda0		= 1.0E-01
lambdaMin	= 1.0E-05
useLowRank	= 0

# INITIAL GUESS CHECK.
max_initial_guess_checks = 0
norm_f0_target = 5.0E-05

# SWEEP CONTROL
rr_phi_max_minimum = 8.0
""" % (dr[i], dr[i], N[i], N[i], N[i-1] + 4, N[i-1] + 4, dr[i-1], dr[i-1], dr[i-1], N[i-1], dr[i-1], N[i-1], dr[i-1], N[i-1], dr[i-1], N[i-1], dr[i-1], N[i-1], dr[i-1], N[i-1], dr[i-1], N[i-1])
    
    f.write(par_file)
    f.close()

In [ ]:
torque_script = """
#!/bin/bash
#
# Name of job.
#PBS -N ROTBOSON,l=2,Resolution_Convergence
#
# Output files.
#PBS -o $PBS_JOBNAME.$PBS_JOBID.out
#PBS -e $PBS_JOBNAME.$PBS_JOBID.err
#
# Set to "short" queue.
#PBS -q short
#
# Resources.
#PBS -l nodes=1:ppn=54
#PBS -l mem=128gb
#PBS -l vmem=146gb
#
# Walltime
#PBS -l walltime=01:00:00
#
# Email notifications.
#PBS -m abe -M santiago.ontanon@correo.nucleares.unam.mx

# Change to current directory.
cd $PBS_O_WORKDIR

# Send email indication job start.
echo -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nExecution has begun." | sendmail santiago.ontanon@correo.nucleares.unam.mx

# Job information.
echo ==================================
echo Executing on : `hostname`
echo Data: `date`
echo Directory: `pwd`
echo Assigned Resources:
echo	`cat $PBS_NODEFILE`
NPROCS=`wc -l < $PBS_NODEFILE`
echo Total: $NPROCS cpus
echo ==================================
cat $PBS_NODEFILE > $HOME/nodos
echo ==================================
echo	Output...
echo ==================================

# Set OMP_NUM_THREADS
export OMP_NUM_THREADS=$PBS_NUM_PPN
export MKL_NUM_THREADS=$PBS_NUM_PPN

# MKL OOC
export MKL_PARDISO_OOC_PATH=/storage/icn/sontanon/ROTBOSON/ooc
export MKL_PARDISO_OOC_MAX_CORE_SIZE=150000
export MKL_PARDISO_OOC_MAX_SWAP_SIZE=10000
export MKL_PARDISO_OOC_KEEP_FILE=1

"""

In [ ]:
for i in range(7, len(N)):
    torque_script += """#time /storage/icn/sontanon/ROTBOSON/out/RConvergence/ROTBOSON "/storage/icn/sontanon/ROTBOSON/out/RConvergence/l=2,w=%.5E,dr=%.5E,N=%04d.par"\n#echo -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nRun %04d has finished." | sendmail santiago.ontanon@correo.nucleares.unam.mx\n\n""" % (0.75, dr[i], N[i], N[i])

In [ ]:
torque_script_name = "/mnt/e/Output17/RConvergence/torque_exe_script.pbs"
f = open(torque_script_name, "w")
f.write(torque_script)
f.close()

In [ ]:
import os

In [ ]:
dirnames = [x[0] for x in os.walk("/mnt/e/Output17/RConvergence/")][1::][::-1][:]
dirnames

In [ ]:
dirnames = [
'/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=1.00000E+00,N=0128',
'/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=8.32797E-01,N=0154',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=6.90667E-01,N=0186',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=5.71744E-01,N=0225',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=4.73492E-01,N=0272',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=3.91831E-01,N=0329',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=3.24969E-01,N=0397',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=2.68951E-01,N=0480',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=2.22700E-01,N=0580',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=1.84342E-01,N=0701',
 '/mnt/e/Output17/RConvergence/l=2,w=7.50000E-01,dr=1.52622E-01,N=0847']

In [ ]:
N

In [ ]:
dr

In [ ]:
k = len(dirnames)

In [ ]:
k

In [ ]:
psi_center = np.zeros(k)
M = np.zeros(k)
J = np.zeros(k)
GRV2 = np.zeros(k)
GRV3 = np.zeros(k)

In [ ]:
for i in range(k):
    dirname = dirnames[i]
    GRV2[i], GRV3[i], psi_center[i], M[i], J[i] = (np.genfromtxt(dirname + "/GRV2.asc").item(), 
     np.genfromtxt(dirname + "/GRV3.asc").item(), 
     np.genfromtxt(dirname + "/sph_psi_f.asc", usecols=0, max_rows=1).item(),
     np.genfromtxt(dirname + "/M_Komar1.asc")[-1],
     np.genfromtxt(dirname + "/J_Komar1.asc")[-1]
    )

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(psi_center[1:] - psi_center[:-1])), ".-", label = r"$\log_{10}|\Delta \psi_0|$")
#ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(GRV2[1:] - GRV2[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_2|$")
#ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(GRV3[1:] - GRV3[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_3|$")
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(M[1:] - M[:-1])), ".-", label = r"$\log_{10}|\Delta M|$")
ax.plot(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(J[1:] - J[:-1])), ".-", label = r"$\log_{10}|\Delta J|$")

ax.set_xlabel(r"$-\log_{10}\,\Delta \rho$")
ax.set_ylabel(r"Difference")

ax.legend()

plt.show()

In [ ]:
from scipy.stats import linregress as linregress

In [ ]:
linregress(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(psi_center[1:] - psi_center[:-1])))

In [ ]:
linregress(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(M[1:] - M[:-1])))

In [ ]:
linregress(-np.log10(np.abs(dr[1:k])), np.log10(np.abs(J[1:] - J[:-1])))

In [ ]:
psi_center_error = (psi_center[-2] - psi_center[-1]) / ((dr[:k][-2] / dr[:k][-1])**4 - 1.0)
psi_center_error

In [ ]:
M_error = (M[-2] - M[-1]) / ((dr[:k][-2] / dr[:k][-1])**4 - 1.0)
M_error

In [ ]:
J_error = (J[-2] - J[-1]) / ((dr[:k][-2] / dr[:k][-1])**4 - 1.0)
J_error

In [ ]:
psi_center[-1], psi_center_error, psi_center_error / psi_center[-1]

In [ ]:
M[-1], M_error, M_error / M[-1]

In [ ]:
J[-1], J_error, J_error / J[-1]

## Memory Use

In [ ]:
mem_use = np.array([4.755, 6.434, 8.615, 12.119, 16.767])

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(np.log10(N[:len(mem_use)]), np.log10(mem_use), ".-", label = r"$\log_{10}|MEM|$")

ax.set_xlabel(r"$\log_{10}\,N$")
ax.set_ylabel(r"$\log_{10}\,MEM$")

ax.legend()

plt.show()

In [ ]:
slope, intercept, rvalue, pvalue, stderr = linregress(np.log10(N[:len(mem_use)]), np.log10(mem_use))

In [ ]:
10**(intercept) * (891)**slope

In [ ]:
for i in range(1, len(N)):
    par_name = "/mnt/e/Output17/BConvergence/l=2,w=7.XXXXXE-01,dr=%.5E,N=%04d.par" % (dr[-2], N[i])
    
    f = open(par_name, "w")
    
    par_file = """
# GRID
dr 		= %.16E
dz 		= %.16E
NrInterior	= %d
NzInterior	= %d
order 		= 4

# SCALAR FIELD PROPERTIES
l	 = 2
m 	 = 1.0

# INITIAL FREQUENCY.
#w0 	= 7.50000E-01

# INITIAL DATA
readInitialData	= 3
NrTotalInitial 	= %d
NzTotalInitial 	= %d
dr_i		= %.16E
dz_i		= %.16E
ghost_i		= 2
order_i		= 4
log_alpha_i 	= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/log_alpha_f.asc"
beta_i		= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/beta_f.asc"
log_h_i		= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/log_h_f.asc"
log_a_i		= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/log_a_f.asc"
psi_i		= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/psi_f.asc"
lambda_i	= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/lambda_f.asc"
w_i		= "/storage/icn/sontanon/ROTBOSON/out/BConvergence/l=2,w=7.4XXXXE-01,dr=%.5E,N=%04d/w_f.asc"

# ANALYTIC INITIAL DATA PARAMETERS.
psi0 	= 1.000
sigmaR	= 0.0
sigmaZ	= 0.0
rExt	= 0.0

# FIXED VARIABLE.
fixedPhi  	= 1
fixedPhiR 	= 2
fixedPhiZ	= 2
fixedOmega 	= 0

# SOLVER PARAMETERS.
solverType	= 1
localSolver	= 1
epsilon		= 2.0E-13
maxNewtonIter   = 50
lambda0		= 1.0E-01
lambdaMin	= 1.0E-05
useLowRank	= 0

# INITIAL GUESS CHECK.
max_initial_guess_checks = 0
norm_f0_target = 5.0E-05

# SWEEP CONTROL
rr_phi_max_minimum = 8.0
""" % (dr[-2], dr[-2], N[i], N[i], N[i-1] + 4, N[i-1] + 4, dr[-2], dr[-2], dr[-2], N[i-1], dr[-2], N[i-1], dr[-2], N[i-1], dr[-2], N[i-1], dr[-2], N[i-1], dr[-2], N[i-1], dr[-2], N[i-1])
    
    f.write(par_file)
    f.close()

In [ ]:
dirnames = [x[0] for x in os.walk("/mnt/e/Output17/BConvergence/")][1::][::][:]
dirnames

In [ ]:
N

In [ ]:
k = len(dirnames)

In [ ]:
k

In [ ]:
rr_inf = (N + 3 - 2 + 0.5) * dr[-2]

In [ ]:
rr_inf

In [ ]:
w = np.zeros(k)
M = np.zeros(k)
J = np.zeros(k)
GRV2 = np.zeros(k)
GRV3 = np.zeros(k)

In [ ]:
for i in range(k):
    dirname = dirnames[i]
    GRV2[i], GRV3[i], w[i], M[i], J[i] = (np.genfromtxt(dirname + "/GRV2.asc").item(), 
     np.genfromtxt(dirname + "/GRV3.asc").item(), 
     np.genfromtxt(dirname + "/w_f.asc", usecols=0, max_rows=1).item(),
     np.genfromtxt(dirname + "/M_Komar1.asc")[-1],
     np.genfromtxt(dirname + "/J_Komar1.asc")[-1]
    )

In [ ]:
fig, ax = plt.subplots(figsize = (8, 6))
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(w[1:] - w[:-1])), ".-", label = r"$\log_{10}|\Delta \omega|$")
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(GRV2[1:] - GRV2[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_2|$")
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(GRV3[1:] - GRV3[:-1])), ".-", label = r"$\log_{10}|\Delta GRV_3|$")
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(M[1:] - M[:-1])), ".-", label = r"$\log_{10}|\Delta M|$")
ax.plot(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(J[1:] - J[:-1])), ".-", label = r"$\log_{10}|\Delta J|$")

ax.set_xlabel(r"$\log_{10}\,r_\infty$")
ax.set_ylabel(r"Difference")

ax.legend()

plt.show()

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(w[1:] - w[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(GRV2[1:] - GRV2[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(GRV3[1:] - GRV3[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(M[1:] - M[:-1])))

In [ ]:
linregress(np.log10(np.abs(rr_inf[1:k])), np.log10(np.abs(J[1:] - J[:-1])))

In [ ]:
rr_inf[-2], rr_inf[-1]

In [ ]:
w_error = (w[-2] - w[-1]) / ((rr_inf[:k][-1] / rr_inf[:k][-2])**2 - 1.0)
w_error

In [ ]:
M_error = (M[-2] - M[-1]) / ((rr_inf[:k][-1] / rr_inf[:k][-2])**2 - 1.0)
M_error

In [ ]:
J_error = (J[-2] - J[-1]) / ((rr_inf[:k][-1] / rr_inf[:k][-2])**2 - 1.0)
J_error

In [ ]:
w[-1], w_error, w_error / w[-1]

In [ ]:
M[-1], M_error, M_error / M[-1]

In [ ]:
J[-1], J_error, J_error / J[-1]